In [2]:
!python -m pip install --upgrade pip --default-timeout=300

In [3]:
!python -m pip install mlflow --timeout 1200 --retries 10 --no-cache-dir

In [4]:
!pip install python-dotenv imblearn awscli boto3

In [6]:
!pip install xgboost

In [3]:
!pip install optuna lightgbm

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [33]:
# !aws configure set aws_access_key_id os.getenv("aws_access_key_id")
# !aws configure set aws_secret_access_key os.getenv("aws_secret_access_key")
# !aws configure set region "ap-south-1"

In [34]:
# import mlflow

# mlflow.set_tracking_uri("http://ec2-13-233-155-25.ap-south-1.compute.amazonaws.com:5000/")

In [35]:
# mlflow.set_experiment("Exp 5 - Selecting best model")

In [5]:
import optuna
from lightgbm import LGBMClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import ADASYN, SMOTE

from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import mlflow.sklearn
import mlflow
import os


In [6]:
df = pd.read_csv("cleaned_df.csv")
df.columns

Index(['Unnamed: 0', 'clean_comment', 'category'], dtype='object')

In [7]:
df.drop(columns=['Unnamed: 0'], inplace=True)
df.isna().sum()

clean_comment    131
category           0
dtype: int64

In [8]:
df.dropna(inplace=True)
df.isnull().sum()

clean_comment    0
category         0
dtype: int64

In [9]:
X = df['clean_comment']
Y = df['category']

In [10]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=.2, random_state=42, stratify=Y)

In [11]:
ngram_range = (1,3)
max_features = 500

vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

In [12]:
x_train_vec.shape

(29329, 500)

In [13]:
x_test_vec.shape

(7333, 500)

In [14]:
#adasyn = ADASYN(random_state=42)
smote = SMOTE(random_state=42)

x_train_vec, y_train = smote.fit_resample(x_train_vec, y_train)

In [19]:
x_train_vec.shape

(37848, 500)

In [15]:
y_train.value_counts()

category
-1    12616
 1    12616
 0    12616
Name: count, dtype: int64

In [16]:
penalty="l2"
C=0.5
solver="saga"
max_iter=1000
random_state=42

lr = LogisticRegression(
    penalty=penalty,
    C=C,
    solver=solver,
    max_iter=max_iter,
    random_state=random_state
)

In [17]:
penalty="l2"
loss="squared_hinge"
C=.5
multi_class="ovr"
class_weight="balanced"
max_iter=2000

learnsvc = LinearSVC(
    penalty=penalty,
    multi_class=multi_class,
    loss=loss,
    C=C,
    max_iter=max_iter,
    class_weight=class_weight,
    random_state=42
)

In [18]:
objective='multi:softprob'  # Outputs class probabilities
num_class=3                 # Explicitly set your 3 classes
    
# Preventing Overfitting on Sparse Text Data
max_depth=5                 # Shallow trees prevent memorizing rare words
learning_rate=0.05       # Balanced step size
n_estimators=200           # Number of boosting rounds
    
    # Feature & Sample Subsampling (Essential for high-dimensional BoW speed)
colsample_bytree=0.5       # Subsample vocabulary per tree
subsample=0.8         # Subsample documents per tree
    
# Technical Parameters
tree_method='hist'          # Uses fast histogram binning (crucial for speed)
n_jobs=-1 


xgb_model = XGBClassifier(
    objective=objective,  # Outputs class probabilities
    num_class=num_class,                 # Explicitly set your 3 classes
    
    # Preventing Overfitting on Sparse Text Data
    max_depth=max_depth,                 # Shallow trees prevent memorizing rare words
    learning_rate=learning_rate,           # Balanced step size
    n_estimators=n_estimators,            # Number of boosting rounds
    
    # Feature & Sample Subsampling (Essential for high-dimensional BoW speed)
    colsample_bytree=colsample_bytree,        # Subsample vocabulary per tree
    subsample=subsample,               # Subsample documents per tree
    
    # Technical Parameters
    tree_method=tree_method,          # Uses fast histogram binning (crucial for speed)
    random_state=42,
    n_jobs=n_jobs                   # Use all available CPU cores
)


In [19]:
n_estimators=200           # Number of trees in the forest
    
# Preventing Overfitting on High-Dimensional Sparse Words
max_depth=20             # Strictly cap depth to save RAM and stop overfitting
max_features='sqrt'         # Randomly samples square root of vocabulary per split
min_samples_split=5         # Prevents building rules for single specific documents
min_samples_leaf=2          # Ensures terminal nodes represent multiple texts
    
    # Optimization Parameters
bootstrap=True           # Keeps trees diverse by sampling data points
n_jobs=-1


rf_model = RandomForestClassifier(
    n_estimators=n_estimators,            # Number of trees in the forest
    
    # Preventing Overfitting on High-Dimensional Sparse Words
    max_depth=max_depth,                # Strictly cap depth to save RAM and stop overfitting
    max_features=max_features,         # Randomly samples square root of vocabulary per split
    min_samples_split=min_samples_split,         # Prevents building rules for single specific documents
    min_samples_leaf=min_samples_leaf,          # Ensures terminal nodes represent multiple texts
    
    # Optimization Parameters
    bootstrap=bootstrap,              # Keeps trees diverse by sampling data points
    n_jobs=n_jobs,                   # Uses all available CPU cores (essential for RF speed)
    random_state=42
)


In [20]:
from sklearn.preprocessing import LabelEncoder

# 1. Initialize and fit the LabelEncoder
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Verify the mapping (it will be: -1 -> 0, 0 -> 1, 1 -> 2)
print("Mapped classes:", le.classes_) 


Mapped classes: [-1  0  1]


In [21]:
models = [lr, learnsvc, xgb_model, rf_model]

for model in models:
    # Gets the readable name of the model for the print statement
    model_name = model.__class__.__name__
    print(f"================== {model_name} ==================")
    
    # 1. Fit using the positive encoded integers (0, 1, 2)
    model.fit(x_train_vec, y_train_encoded)
    y_pred_encoded = model.predict(x_test_vec)

    # 2. Transform predictions back to original labels (-1, 0, 1) for evaluation
    y_pred = le.inverse_transform(y_pred_encoded)

    # 3. Calculate and print performance
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nAccuracy: {accuracy:.4f}\n")

    classification_rep = classification_report(y_test, y_pred)
    print(f"Classification Report:\n{classification_rep}")
    print("====================================================\n")


================== LogisticRegression ==================

Accuracy: 0.7233

Classification Report:
              precision    recall  f1-score   support

          -1       0.59      0.57      0.58      1650
           0       0.70      0.87      0.78      2529
           1       0.83      0.68      0.75      3154

    accuracy                           0.72      7333
   macro avg       0.71      0.71      0.70      7333
weighted avg       0.73      0.72      0.72      7333


================== LinearSVC ==================

Accuracy: 0.7329

Classification Report:
              precision    recall  f1-score   support

          -1       0.64      0.55      0.59      1650
           0       0.69      0.91      0.79      2529
           1       0.84      0.69      0.76      3154

    accuracy                           0.73      7333
   macro avg       0.72      0.72      0.71      7333
weighted avg       0.74      0.73      0.73      7333


================== XGBClassifier ==============

In [28]:
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 1000)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int("max_depth", 3, 15)
    num_leaves = trial.suggest_int("num_leaves",20, 150)
    min_child_samples = trial.suggest_int("min_child_samples", 10,100)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-4, 10, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-4, 10, log=True)

    lightgbm_model = LGBMClassifier(
        n_estimators = n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        colsample_bytree=colsample_bytree,
        subsample=subsample,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42
    )
    lightgbm_model.fit(x_train_vec, y_train)
    #y_pred_encoded = lightgbm_model.predict(x_test_vec)
    #y_pred = le.inverse_transform(y_pred_encoded)
    y_pred = lightgbm_model.predict(x_test_vec)
    accuracy = accuracy_score(y_test,y_pred)

    return accuracy

In [29]:
def log_mlflow(model_name, model, params, x_train_vec, x_test_vec, y_train, y_test):
    with mlflow.start_run():
        mlflow.log_param("Model Name", model_name)
        for name, value in params.items():
            mlflow.log_param(name, value=value)

        model.fit(x_train_vec, y_train)
        y_pred = model.predict(x_test_vec)
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        mlflow.sklearn.log_model(
    model,
    name="LGBMClassifier_model",
    skops_trusted_types=[
        "sklearn.tree._tree.Tree",
        "collections.OrderedDict",
        "lightgbm.basic.Booster",
        "lightgbm.sklearn.LGBMClassifier",
    ],
)


In [30]:
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=50)

    best_params = study.best_params

    params = {
        "n_estimators": best_params['n_estimators'],
        "learning_rate": best_params['learning_rate'],
        "max_depth": best_params['max_depth'],
        "num_leaves": best_params['num_leaves'],
        "min_child_samples": best_params['min_child_samples'],
        "colsample_bytree": best_params['colsample_bytree'],
        "subsample": best_params['subsample'],
        "reg_alpha": best_params['reg_alpha'],
        "reg_lambda": best_params['reg_lambda'],
        "random_state": 42
    }


    best_model = LGBMClassifier(
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        max_depth=best_params['max_depth'],
        num_leaves=best_params['num_leaves'],
        min_child_samples=best_params['min_child_samples'],
        colsample_bytree=best_params['colsample_bytree'],
        subsample=best_params['subsample'],
        reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'],
        random_state=42
    )

    best_model.fit(x_train_vec, y_train)
    #y_pred_encoded = best_model.predict(x_test_vec)
    #y_pred = le.inverse_transform(y_pred_encoded)
    y_pred = best_model.predict(x_test_vec)

    accuracy = accuracy_score(y_test, y_pred)
    classification_rep = classification_report(y_test, y_pred)

    print(f"============================================\naccuracy: {accuracy}\n")
    print(f"============================================\nClassification report: {classification_rep}\n")

    optuna.visualization.plot_param_importances(study=study).show()
    optuna.visualization.plot_optimization_history(study).show()

    print("\n===== mlflow process initializing ======")
    log_mlflow(best_model.__class__.__name__,  best_model, params, x_train_vec, x_test_vec, y_train, y_test)

In [26]:
pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.


In [32]:
run_optuna_experiment()

[I 2026-09-21 01:31:57,189] A new study created in memory with name: no-name-ea7dc97e-2900-4c85-b037-48e9ed4cd2bd


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.067183 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 73722
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 495
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

[I 2026-09-21 01:32:23,549] Trial 0 finished with value: 0.6513023319241784 and parameters: {'n_estimators': 370, 'learning_rate': 0.0007097218381057853, 'max_depth': 14, 'num_leaves': 90, 'min_child_samples': 22, 'colsample_bytree': 0.8831234075207874, 'subsample': 0.6308493823664656, 'reg_alpha': 0.01738866568602109, 'reg_lambda': 0.823721600521904}. Best is trial 0 with value: 0.6513023319241784.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033165 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 73628
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 489
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

[I 2026-09-21 01:32:33,101] Trial 1 finished with value: 0.6095731624164735 and parameters: {'n_estimators': 687, 'learning_rate': 0.00016111833334230664, 'max_depth': 5, 'num_leaves': 128, 'min_child_samples': 85, 'colsample_bytree': 0.5071312531532022, 'subsample': 0.8891999095285934, 'reg_alpha': 5.158090467497146, 'reg_lambda': 0.0002999523264617694}. Best is trial 0 with value: 0.6513023319241784.


...

[I 2026-09-21 01:49:08,145] Trial 48 finished with value: 0.7410336833492431 and parameters: {'n_estimators': 665, 'learning_rate': 0.03622085084521205, 'max_depth': 12, 'num_leaves': 25, 'min_child_samples': 16, 'colsample_bytree': 0.5537848535256995, 'subsample': 0.5533588722751034, 'reg_alpha': 1.8886924102033673, 'reg_lambda': 5.355165786128376}. Best is trial 43 with value: 0.7419882721941906.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.037676 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 73722
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 495
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

[I 2026-09-21 01:49:31,274] Trial 49 finished with value: 0.7429428610391381 and parameters: {'n_estimators': 760, 'learning_rate': 0.023697402540973824, 'max_depth': 13, 'num_leaves': 29, 'min_child_samples': 19, 'colsample_bytree': 0.7635445530698962, 'subsample': 0.576130750681706, 'reg_alpha': 0.5750750699968362, 'reg_lambda': 2.7376877403301547}. Best is trial 49 with value: 0.7429428610391381.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.061777 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 73722
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 495
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s


===== mlflow process initializing ======
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.054206 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 73722
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 495
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [24]:
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 1000)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int("max_depth", 3, 15)
    num_leaves = trial.suggest_int("num_leaves",20, 150)
    min_child_samples = trial.suggest_int("min_child_samples", 10,100)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-4, 10, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-4, 10, log=True)

    lightgbm_model = LGBMClassifier(
        n_estimators = n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        colsample_bytree=colsample_bytree,
        subsample=subsample,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42
    )

    